

## The job

A restaurant owner has hired you for the week. She has two files sitting on her laptop and no
idea what to do with either of them.

- **`tips.csv`** — the till export. Every bill her restaurant has taken, with the tip.
- **`bookings.csv`** — a table reservation log her staff typed in by hand.

She wants a summary she can actually read, and she is going to keep asking follow-up
questions. Every new tool today shows up because the previous step ran into a wall — not
because it is next on a list.

By the end you will have built her report, saved it, broken it, fixed it, and cleaned up the
booking log.

In [ ]:
import pandas as pd
tips = pd.read_csv('/Users/shailesh/Desktop/vedam_Sem3_ml/Data/tip.csv')
print(tips.shape)
tips.head()

---
# Part A — Build her the report

## 1. "Just give me one table"

She wants three things per day: how many bills, the average bill, the average tip.

You already know how to get all three. Watch what it looks like.

In [ ]:
tips.groupby("day")["total_bill"].size()

day
Fri     19
Sat     87
Sun     76
Thur    62
Name: total_bill, dtype: int64

In [12]:
tips.groupby("day")["total_bill"].mean()
tips.groupby("day")["tip"].mean()

day
Fri     2.734737
Sat     2.993103
Sun     3.255132
Thur    2.771452
Name: tip, dtype: float64

Three separate tables. Three column names you did not choose — every one of them says
`total_bill` or `tip`. Nothing here can be emailed to anybody.

She asked for **one table**. So build one.

### Named aggregations

```python
df.groupby('key').agg(
    column_name_you_pick = ('column_to_use', 'function_to_apply'),
    ...
)
```

Each keyword on the left becomes a column heading. Each tuple on the right reads as
*"take this column, apply this function"*.

In [13]:
tips.groupby("day").agg(
    Bill_count=("total_bill",'size'),
    avg_bill=("total_bill",'mean') ,
    avg_tip=('tip','mean')
)

,Bill_count,avg_bill,avg_tip
day,,,
Fri,19,17.151579,2.734737
Sat,87,20.441379,2.993103
Sun,76,21.410000,3.255132
Thur,62,17.682742,2.771452


One table. Headings in plain English. This is the form to reach for by default.

**Functions you can pass as strings:** `'size'`, `'count'`, `'sum'`, `'mean'`, `'median'`,
`'min'`, `'max'`, `'std'`, `'nunique'`, `'first'`, `'last'`.

> **Watch the tuple order.** `('total_bill', 'mean')` is right.
> `('mean', 'total_bill')` makes pandas hunt for a column called `mean`.

### Your turn — Q1

She also wants the same breakdown by **service** rather than by day: Lunch versus Dinner.

Build one table with `bills`, `avg_bill`, and `avg_party` — the average number of people at
the table. *(The party size column is called `size`. Yes, the same word as the function.
Read the tuple carefully.)*

In [ ]:
# your code here


---
## 2. "Only show me the good days"

She looks at your table and says: *drop the days where the average tip is under 3.*

Easy — you have been filtering since day one.

In [15]:
summary=tips.groupby("day").agg(
    Bill_count=("total_bill",'size'),
    avg_bill=("total_bill",'mean') ,
    avg_tip=('tip','mean')
).round(2)
summary

,Bill_count,avg_bill,avg_tip
day,,,
Fri,19,17.15,2.73
Sat,87,20.44,2.99
Sun,76,21.41,3.26
Thur,62,17.68,2.77


In [17]:
summary[summary["day"]=="Sun"]

KeyError: 'day'

**KeyError: 'day'.**

But `day` is right there on the screen. You can see it. So where has it gone?

In [22]:
print(summary.columns.tolist())
print(summary.index.to_list())


['Bill_count', 'avg_bill', 'avg_tip']
['Fri', 'Sat', 'Sun', 'Thur']


There it is. After a `groupby`, **the grouping key stops being a column and becomes the
index.** It prints on the left edge, which makes it look like a column, but
`summary['day']` cannot find it because it is not in `.columns`.

### `reset_index()`

Pushes the index back into a normal column and hands you a plain 0, 1, 2… index instead.

In [24]:
flat=summary.reset_index()
print(flat.columns.tolist())
flat.head()

['day', 'Bill_count', 'avg_bill', 'avg_tip']


,day,Bill_count,avg_bill,avg_tip
0,Fri,19,17.15,2.73
1,Sat,87,20.44,2.99
2,Sun,76,21.41,3.26
3,Thur,62,17.68,2.77


> **Rule of thumb.** If the next thing you want to do is *filter, merge, plot, or save* —
> `reset_index()` first. If you are only looking at the table, leave it.

It behaves the same way with two grouping keys: both come back as columns.

### Your turn — Q2

Build a summary of `bills`, `avg_bill` and `avg_tip` per **day**, then show only the days
where the average bill is above 20.

You will need two steps. If you get a `KeyError`, you skipped one.

In [ ]:
# your code here


---
## 3. "Email me the file"

She wants the summary as a CSV. You have seen `to_csv` before, and you have seen
`index=False` in every tutorial you have ever read, so you type it without thinking.

In [25]:
summary

,Bill_count,avg_bill,avg_tip
day,,,
Fri,19,17.15,2.73
Sat,87,20.44,2.99
Sun,76,21.41,3.26
Thur,62,17.68,2.77


In [28]:
summary.to_csv("Report.csv")
flat=pd.read_csv("Report.csv")
flat.head()

,day,Bill_count,avg_bill,avg_tip
0,Fri,19,17.15,2.73
1,Sat,87,20.44,2.99
2,Sun,76,21.41,3.26
3,Thur,62,17.68,2.77


Look at that file. Four rows of numbers. **Every one of them is correct.**

And there is nothing at all to say which row is Thursday and which is Sunday.

No error. No warning. The file was created, it opens fine in Excel, and the work is gone.
This is the same problem as five minutes ago wearing a different coat: `day` lives in the
**index**, and `index=False` means *do not write the index*.

Both of these keep the days. Version C is better — every column has a proper heading, and
reading it back gives you an ordinary DataFrame with no surprises.

> **Rule.** `index=False` is correct for a *raw* DataFrame, where the index is just 0, 1, 2…
> and means nothing. It is wrong for a *grouped* result, where the index is your data.
> `reset_index()` first, and then `index=False` is always safe.

Read it back. It takes four seconds and it is the only way to be sure.

### Your turn — Q3

She now wants a smoking-section report: per `smoker` group, the number of bills, average bill
and average tip. Save it as `smoker_report.csv` so that the Yes/No labels survive.

Then read your own file back and check.

In [ ]:
# your code here
smoker_report=tips.groupby('smoker').agg(
    bill_count=("total_bill",'size'),
    avg_bill=("total_bill","mean"),
    avg_tip=("tip",'mean')

)

smoker_report.reset_index().to_csv("smoker_report.csv",index=False)
temp=pd.read_csv("smoker_report.csv")
temp.head()


,smoker,bill_count,avg_bill,avg_tip
0,No,151,19.188278,2.991854
1,Yes,93,20.756344,3.008710


---
## 4. "Who left that huge tip?"

She reads the report and spots something.

*"Ten pounds on a Saturday — who was that? What did they order? Was it a big group?"*

You cannot tell her. `.max()` returned the number and threw away everything else about that
row. You know a 10.00 tip exists. You know nothing about the table that left it.

### `idxmax()`

Returns the **index label of the row** holding the maximum — not the value, the *address*.

In [40]:
# print(tips.groupby("day")['tip'].idxmax())
# print(tips.groupby("day")['tip'].max())
print(tips.loc[170])

total_bill     50.81
tip             10.0
sex             Male
smoker           Yes
day              Sat
time          Dinner
size               3
Name: 170, dtype: object


In [37]:
tips.loc[183]

total_bill     23.17
tip              6.5
sex             Male
smoker           Yes
day              Sun
time          Dinner
size               4
Name: 183, dtype: object

Those are row positions, not tips. Feed them to `.loc[]` and the whole rows come back.

day
Fri      4.73
Sat     10.00
Sun      6.50
Thur     6.70
Name: tip, dtype: float64

Now you can answer her. The 10.00 Saturday tip came from a **party of 3 on a 50.81 bill** —
just under 20%. Generous, but the size of the bill is doing most of the work, not unusual
kindness.

The two-step pattern is worth memorising:

```python
rows = df.groupby('key')['value'].idxmax()   # step 1 — which rows
df.loc[rows]                                 # step 2 — go get them
```

`idxmin()` is identical except for direction.

> **Two edge cases.** `idxmax()` raises an error if every value in a group is missing, and on
> a tie it silently returns only the *first* match.

### Your turn — Q4

She wants to find the rudest tables. *"Which table on each day left the smallest tip
relative to their bill?"*

Note carefully: **relative to their bill.** The smallest tip in rupees is not the same
question as the stingiest tip.

1. Add a `tip_pct` column — tip as a percentage of the bill.
2. Find the worst table on each day, showing the bill, the tip and the percentage.
3. Then run the same thing with `idxmax` and look hard at the Sunday row before you report it
   to her as your best customer.

In [ ]:
# your code here



---
# Part B — The booking log

Part A was the till export: machine-generated, clean, every column exactly what it claimed to
be. Real data is rarely like that.

The booking log was typed by hand by three different members of staff over several months.

In [42]:
bookings = pd.read_csv('/Users/shailesh/Desktop/vedam_Sem3_ml/Data/bookings.csv')
bookings

,booking_id,customer,day,time,party_size
0,101,ravi sharma,Sat,Dinner,4.0
1,102,PRIYA IYER,Sat,Dinner,2.0
2,103,Ravi Sharma,Sun,Lunch,NaN
3,104,amit kumar,Sun,Dinner,6.0
4,105,priya iyer,Fri,Dinner,NaN
5,106,RAVI SHARMA,Sat,Lunch,3.0
6,107,sana khan,Sat,Lunch,NaN
7,108,Amit Kumar,Sat,Dinner,5.0
8,109,vikram rao,Sun,Dinner,2.0
9,110,Sana Khan,Fri,Dinner,4.0


In [43]:
bookings['customer'].nunique()

12

Twelve rows. Small enough to read every single one — do that now, before running anything.

Two things should bother you.

---
## 5. "How many regulars do we have?"

Reasonable question. Reasonable answer:

**Twelve customers in twelve bookings.** Nobody has ever come back. She should probably close.

Except look at the actual names again. `ravi sharma`, `RAVI SHARMA`, `  Ravi Sharma `.
That is one man who has eaten there three times, and pandas is counting him as three
different people, because to a computer those really are three different strings.

### The `.str` accessor

Numeric columns give you `.mean()` and `.round()`. Text columns give you nothing — until you
go through `.str`, which applies an ordinary Python string method to every row at once.

In [ ]:
print(bookings['customer'].str.lower().str.strip())

0     ravi sharma
1      priya iyer
2     ravi sharma
3      amit kumar
4     priya  iyer
5     ravi sharma
6       sana khan
7      amit kumar
8      vikram rao
9      sana  khan
10     priya iyer
11     vikram rao
Name: customer, dtype: object


In [ ]:
bookings['customer']=bookings['customer'].str.lower() \
                .str.strip().str.split().str.join(" ")

In [ ]:
print(bookings['customer'].nunique())
print(bookings['customer'].unique().tolist())


5
['ravi sharma', 'priya iyer', 'amit kumar', 'sana khan', 'vikram rao']


Twelve down to **seven**. Better. But she has five tables' worth of regulars in her head, not
seven. Count again, carefully.

`priya iyer` and `priya  iyer`. `sana khan` and `sana  khan`.

**Double space in the middle.** `.strip()` only removes whitespace from the *ends* of a
string — it does not touch the middle. This is the kind of thing you will never see by
looking at printed output, because two spaces and one space look nearly identical on screen.

The fix reuses `.split()`, which breaks on *any* run of whitespace, and then rejoins with
exactly one:

**Five customers.** Ravi and Priya have each been in three times.

Twelve regulars or five is not a rounding difference — it is the difference between a
restaurant with no repeat business and one with a loyal core. That decision came out of two
`.str` calls.

### The rest of `.str`

| Method | Does |
|--------|------|
| `.str.lower()` / `.str.upper()` | Case-fold — always do this before comparing text |
| `.str.strip()` | Trim the **ends** only |
| `.str.split()` + `.str[i]` | Break apart, then take one piece |
| `.str.split().str.join(' ')` | Collapse internal whitespace |
| `.str.contains(x, na=False)` | Boolean mask for filtering |
| `.str.replace(a, b)` | Swap a substring |
| `.str.startswith()` / `.endswith()` | Prefix or suffix match |

### Your turn — Q5

1. Split `customer` into a `first_name` column.
2. How many bookings has each first name made?
3. Now a design question, not a syntax one: her staff wants to switch to storing only first
   names to save typing. Using this data, tell her why that is a bad idea.

In [ ]:
# your code here


---
## 6. "What's our average table size?"

The second thing that should have bothered you in the raw log: some `party_size` cells are
empty. Three of them.

She wants the average anyway.

3.62. Fine. But **3.62 out of how many bookings?**

### `.agg()` with a list, and the `size` / `count` trap

`.agg()` also takes a plain list of functions — same column, several summaries at once.

| Function | Counts |
|----------|--------|
| `size` | **Rows** in the group — blanks included |
| `count` | **Non-missing values** in that column — blanks excluded |

Twelve bookings. Eight party sizes recorded. That 3.62 average was computed from **eight
rows, not twelve** — pandas dropped the blanks without mentioning it.

If a column has no blanks, `size` and `count` agree and you will never notice the
difference. The moment it has blanks they diverge, and if you wrote `count` while meaning
"how many bookings", your denominator is quietly wrong.

Read that table across. Friday: **two** bookings, **one** party size recorded. That Friday
average is one number wearing a suit.

> **Habit worth forming.** Whenever you report a group mean, put `size` and `count` beside
> it. If they differ, whoever reads your table deserves to know how much of the group the
> number actually describes.

Named aggregation gives you the same thing with honest labels:

Twelve rows made this easy to see, because you could check it by eye. It does not go away at
scale — it just gets harder to spot.

891 passengers aboard. **714 ages recorded.** Any average age you compute silently ignores
177 people, so your "average passenger" is really "average passenger *whose age somebody
wrote down*" — and whether those two are the same thing depends on why the age is missing.
It usually is not random.

### Your turn — Q6

She wants to fix the booking form, and needs to know which shift is worst at filling it in.

1. For each `day`, how many bookings are there, how many have a party size, and how many are
   missing?
2. Which day is worst — by **count** of missing, and by **percentage**?
3. The two answers disagree. Which one would you actually tell her, and why?

In [68]:
x=tips['day']
print(x)

print(x.reset_index(name="day"))

0       Sun
1       Sun
2       Sun
3       Sun
4       Sun
       ... 
239     Sat
240     Sat
241     Sat
242     Sat
243    Thur
Name: day, Length: 244, dtype: object
     index   day
0        0   Sun
1        1   Sun
2        2   Sun
3        3   Sun
4        4   Sun
..     ...   ...
239    239   Sat
240    240   Sat
241    241   Sat
242    242   Sat
243    243  Thur

[244 rows x 2 columns]
